# Graph Summarization Algorithm

This notebook demonstrates a simplified DAG-based graph summarization algorithm applied to a CFG tha should be effectively the same as summarize_graph.py.

For simplicity, this doesn't apply the existing pruning step, but it should work with or without.

## Algorithm Overview

1. Initialize empty summaries for all functions/communities
2. Use GreedyFAS to remove the minimum number of edges to create an acyclic graph (DAG) (using `nx.minimum_feedback_arc_set`), which is required to compute a topological sort.
3. Compute reverse topological sort to determine processing order (using `nx.topological_sort`) 
4. Process functions sequentially. The topological sort step (3) guarantees that dependencies are summarized first, so they can be used as context for subsequent summarizations.


In [3]:
import networkx as nx
import matplotlib.pyplot as plt
from ipysigma import Sigma
import numpy as np
from collections import defaultdict

## 0. Load CFG

In [6]:
from src.visualize_cfg import load_cfg

G = load_cfg('reorder_and_pad2.json')

## 1. Detect and Aggregate Communities

For now I'm just grouping nodes by function, and sorting each block's assembly by address.

In [7]:
def create_function_communities(G):
    """Group basic blocks by their function and create a function-level graph"""
    import networkx as nx
    
    # Group nodes by function
    func_nodes = {}
    for node, data in G.nodes(data=True):
        func = data.get("func", str(node))
        if func not in func_nodes:
            func_nodes[func] = []
        func_nodes[func].append(node)
    
    # Create function-level graph
    G_funcs = nx.DiGraph()
    
    # Add function nodes with aggregated data
    for func_name, nodes in func_nodes.items():
        # Sort nodes by their address (node ID or label) to maintain logical order
        sorted_nodes = sorted(nodes, key=lambda n: (G.nodes[n].get("label", str(n))))
        
        # Collect all instructions and labels from this function's blocks in order
        all_instrs = []
        all_labels = []
        for node in sorted_nodes:
            node_data = G.nodes[node]
            all_instrs.extend(node_data.get("instrs", []))
            all_labels.append(node_data.get("label", str(node)))
        
        G_funcs.add_node(func_name, 
                         func=func_name,
                         label=func_name,
                         instrs=all_instrs,
                         block_labels=all_labels)
    
    # Add edges between functions
    seen_edges = set()
    for src, dst, edata in G.edges(data=True):
        src_func = G.nodes[src].get("func", str(src))
        dst_func = G.nodes[dst].get("func", str(dst))
        
        # Don't add self-loops or duplicate edges
        if src_func != dst_func and (src_func, dst_func) not in seen_edges:
            edge_type = edata.get("type", "unknown")
            G_funcs.add_edge(src_func, dst_func, type=edge_type)
            seen_edges.add((src_func, dst_func))
    
    return G_funcs, list(func_nodes.keys())

G_communities, community_list = create_function_communities(G)

widget = Sigma(
    G_communities,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in n else d.get('color', 'blue'),
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)

Sigma(nx.DiGraph with 343 nodes and 749 edges)

## 2. Remove Cycles using GreedyFAS

The minimum feedback arc set problem (FAS) is NP-hard so tihs is a vibe-coded implementation of GreedyFAS ([paper](https://arxiv.org/pdf/2208.09234)), an approximate version. It reomves the minimum number of edges in a directed graph to make it acyclic.



In [8]:
import networkx as nx

def greedy_feedback_arc_set(G):
    feedback_edges = set()
    visited = set()
    rec_stack = set()
    
    def dfs(node):
        visited.add(node)
        rec_stack.add(node)
        
        for neighbor in G.successors(node):
            if neighbor not in visited:
                dfs(neighbor)
            elif neighbor in rec_stack:
                # Back edge found - add to feedback arc set
                feedback_edges.add((node, neighbor))
        
        rec_stack.remove(node)
    
    for node in G.nodes():
        if node not in visited:
            dfs(node)
    
    return feedback_edges

# Apply greedy FAS
fas_edges = greedy_feedback_arc_set(G_communities)

# Create DAG by removing FAS edges
G_dag = G_communities.copy()
G_dag.remove_edges_from(fas_edges)

widget = Sigma(
    G_dag,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)

print(f"FAS size: {len(fas_edges)} edges")
print(f"Before FAS removal: {G_communities.number_of_nodes()} nodes, {G_communities.number_of_edges()} edges")
print(f"After FAS removal: {G_dag.number_of_nodes()} nodes, {G_dag.number_of_edges()} edges")
    

Sigma(nx.DiGraph with 343 nodes and 696 edges)

FAS size: 53 edges
Before FAS removal: 343 nodes, 749 edges
After FAS removal: 343 nodes, 696 edges


## 3. Compute Reverse Topological Sort

Now that the graph is a DAG, we can find a topological sort and simply summarize them in reverse order. This guarantees that every node's dependencies/children are summarized before the node itself, without any tricky graph traversal stuff (e.g. checking if a node has been summarized).

In [9]:
summary_order = list(reversed(list(nx.topological_sort(G_dag))))
print(summary_order)

['sub_14001266C', 'sub_140016D8C', 'sub_140014EB0', 'sub_140011FA4', 'sub_1400124BC', 'sub_140011760', 'sub_140016E48', 'sub_140014CC4', 'sub_1400127C4', 'sub_1400127B4', 'sub_140004B90', 'sub_1400135B8', 'sub_140014940', 'sub_1400150A4', 'sub_140016CC8', 'sub_140013BCC', 'sub_140011494', 'sub_140016C8C', 'sub_140014EDC', 'sub_140012B80', 'sub_140016C3C', 'sub_140011134', 'sub_140002C7C', 'sub_140012B6C', 'sub_140016C2C', 'sub_140012400', 'sub_140001040', 'sub_140012C2C', 'sub_140012FFC', 'sub_140012EC8', 'sub_14001171C', 'sub_1400127E0', 'sub_14000B478', 'memcmp', '__alloca_probe', 'sub_1400013BC', 'sub_140013304', 'sub_14000FD78', 'sub_14000AF24', 'sub_14000AF10', '_ZN4core3str8converts9from_utf817he763c5b9963ab195E', 'sub_14000CEE0', 'sub_14000B048', 'sub_140014054', 'sub_140016BF4', 'sub_140005564', 'sub_1400061A8', 'sub_14000D1A4', 'sub_14000EFD0', 'sub_14000CB1C', 'sub_1400127D4', 'sub_14000FA0C', 'sub_14000CD90', 'sub_140005768', 'sub_140013E4C', 'sub_14000BCA8', 'sub_14000BBF4'

## 4. Summarize with LLM in Reverse Topological Order

Now we can simply loop through the list of nodes sequentially, and know that for each node we will have as many dependencies summarized as possible by that iteration. We add the LLM output to the summary attribute of the **original** graph with all original control flow edges, and save it at the end.

*Ideally, we would save as we go, or do some kind of checkpointing, but this notebook is just for demonstrating the idea anyway.*

In [ ]:
import ollama
from tqdm import tqdm
import json

# Updated System prompts - more specific about what we need
FUNC_SYSTEM_PROMPT = (
    "You are a reverse-engineering assistant analyzing aarch64 assembly.\n"
    "TASK: Provide a concise, specific summary of what this function computes.\n\n"
    "Your summary MUST answer:\n"
    "1. What are the inputs? (parameters passed in x0-x7)\n"
    "2. What does the function compute? (specific operations, not generic 'processing')\n"
    "3. What is returned? (in x0)\n\n"
    "Be specific. Instead of 'processes data', say 'computes a checksum' or 'sorts an array'. "
    "Don't describe standard calling conventions. "
    "Assume your audience is a reverse engineer who needs to know what this function does without reading assembly."
)

FUNC_WITH_DEPS_SYSTEM_PROMPT = (
    "You are a reverse-engineering assistant analyzing aarch64 assembly.\n"
    "TASK: Provide a concise, specific summary of what this function does, incorporating its dependencies.\n\n"
    "Your summary MUST answer:\n"
    "1. What are the inputs? (parameters and what they represent)\n"
    "2. What is the purpose? (be specific: 'validates X', 'allocates and initializes Y', etc.)\n"
    "3. How does it use the called functions? (not their internal details, just their role here)\n"
    "4. What is returned? (in x0 and what it represents)\n\n"
    "Be specific and concrete. Avoid generic phrases like 'data processing' or 'initialization'. "
    "Assume callers want to know: 'Should I call this before X?' or 'Does this allocate memory?' "
    "Don't describe standard calling conventions.\n"
    "If a called function's summary is missing, just note what it's called with."
)

# Configure LLM
model = "qwen2.5-coder:7b" 
base_url = "http://100.104.79.110:11434"
client = ollama.Client(host=base_url)

# Process all nodes
for node in tqdm(summary_order):
    successors = list(G_dag.successors(node))
    
    # Get the actual assembly instructions and block labels
    instrs = G_communities.nodes[node].get('instrs', [])
    block_labels = G_communities.nodes[node].get('block_labels', [])
    
    # Format instructions for the prompt
    if isinstance(instrs, list):
        instrs_str = '\n'.join(instrs)
    else:
        instrs_str = str(instrs)
    
    # Limit instruction length to avoid token explosion (keep first 2000 chars)
    if len(instrs_str) > 2000:
        instrs_str = instrs_str[:2000] + "\n... (truncated)"
    
    # Build dependency summaries with edge type info
    deps_text = ""
    for succ in successors:
        succ_summary = G_communities.nodes[succ].get('summary', "")
        edge_type = G_dag[node][succ].get('type', 'unknown')
        if succ_summary:
            deps_text += f"\n[{edge_type}] {succ}:\n{succ_summary}\n"
    
    # Construct the task prompt with actual assembly code
    if deps_text:
        user_content = (
            f"Function: {node}\n"
            f"Basic Blocks: {', '.join(block_labels)}\n\n"
            f"Assembly Code:\n{instrs_str}\n\n"
            f"Called/Referenced Functions:\n{deps_text}"
        )
        system_prompt = FUNC_WITH_DEPS_SYSTEM_PROMPT
    else:
        user_content = (
            f"Function: {node}\n"
            f"Basic Blocks: {', '.join(block_labels)}\n\n"
            f"Assembly Code:\n{instrs_str}"
        )
        system_prompt = FUNC_SYSTEM_PROMPT
    
    response = client.generate(model=model, prompt=f"{system_prompt}\n\n{user_content}", stream=False)
    summary = response["response"]
    G_communities.nodes[node]['summary'] = summary

# Save summaries
summaries = {node: G_communities.nodes[node]['summary'] for node in G_communities.nodes() if 'summary' in G_communities.nodes[node]}
with open('summaries.json', 'w') as f:
    json.dump(summaries, f, indent=2)
print(f"Summarization complete. Saved {len(summaries)} summaries to summaries.json")

100%|██████████| 343/343 [35:16<00:00,  6.17s/it]

Summarization complete. Saved 343 summaries to summaries.json


## Hierarchal Approach

This could be applied in a hierarchal way, but detecting and aggregating communities may re-introduce cycles, so the GreedyFAS edge removal step will have to be applied at each level.